# 02 — Training Demo
Run the full training pipeline and observe live loss/accuracy updates.

In [1]:
import sys
sys.path.insert(0, '..')

from utils.config import load_config
from utils.helpers import set_seed, get_device
from data.dataset_loader import build_dataloaders
from data.preprocessing import get_train_transform, get_val_transform
from models.vl_jepa_model import VLJEPAModel
from training.trainer import Trainer
from utils.visualization import plot_training_history
import matplotlib.pyplot as plt
import os

print('Imports OK')

Imports OK


In [2]:
# ── Config ────────────────────────────────────────────────────────
cfg = load_config('../configs')

# Override for notebook (fewer epochs for quick demo)
cfg['training']['epochs']  = 10
cfg['training']['batch_size'] = 16

set_seed(cfg['project']['seed'])
device = get_device(cfg['device'])
print(f'Device: {device}')

Device: mps


In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────
# ⚠️ Use ONLY train/ for the 70/15/15 split — test/ stays as true holdout
data_root = os.path.join('..', cfg['paths']['data_root'], 'augmented_alzheimer_mri_dataset')

train_loader, val_loader, test_loader, vocab_size = build_dataloaders(
    data_root       = data_root,
    train_transform = get_train_transform(cfg['dataset']['image_size']),
    val_transform   = get_val_transform(cfg['dataset']['image_size']),
    batch_size      = cfg['training']['batch_size'],
    num_workers     = 0,
    max_seq_len     = cfg['text_encoder']['max_seq_len'],
)

print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')
print(f'Vocab size: {vocab_size}')


[dataset] '../data/raw/train/augmented_alzheimer_mri_dataset' does not contain a train/ folder. Searching for the dataset automatically...


FileNotFoundError: No dataset directories found under '../data/raw/train'.
Expected sub-folders like AugmentedAlzheimerDataset/ or OriginalDataset/
Download with: bash scripts/download_dataset.sh

In [5]:
# ── Model ─────────────────────────────────────────────────────────
model = VLJEPAModel(
    vocab_size     = vocab_size,
    embedding_dim  = cfg['image_encoder']['embedding_dim'],
    projection_dim = cfg['vl_jepa']['projection_dim'],
    num_classes    = cfg['vl_jepa']['num_classes'],
    dropout        = cfg['vl_jepa']['dropout'],
    use_text       = cfg['vl_jepa']['use_text_branch'],
)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total:,}')

Trainable parameters: 639,716


In [6]:
# ── Train ─────────────────────────────────────────────────────────
trainer = Trainer(
    model            = model,
    train_loader     = train_loader,
    val_loader       = val_loader,
    device           = device,
    lr               = cfg['training']['learning_rate'],
    weight_decay     = cfg['training']['weight_decay'],
    epochs           = cfg['training']['epochs'],
    checkpoint_dir   = os.path.join('..', cfg['paths']['checkpoint_dir']),
    use_amp          = cfg['training']['use_amp'],
    early_stopping_patience = cfg['training']['early_stopping']['patience'],
)

history = trainer.fit()

Epoch [001/10]  Train Loss: 0.2735  Acc: 0.997 | Val Loss: 0.0000  Acc: 1.000  (214.3s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [002/10]  Train Loss: 0.2567  Acc: 1.000 | Val Loss: 0.0000  Acc: 1.000  (208.2s)


Epoch [003/10]  Train Loss: 0.2442  Acc: 1.000 | Val Loss: 0.0000  Acc: 1.000  (212.8s)


Epoch [004/10]  Train Loss: 0.2349  Acc: 1.000 | Val Loss: 0.0000  Acc: 1.000  (217.5s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [005/10]  Train Loss: 0.2283  Acc: 1.000 | Val Loss: 0.0000  Acc: 1.000  (215.6s)


Epoch [006/10]  Train Loss: 0.2232  Acc: 1.000 | Val Loss: 0.0000  Acc: 1.000  (209.3s)


Epoch [007/10]  Train Loss: 0.2188  Acc: 1.000 | Val Loss: 0.0000  Acc: 1.000  (209.5s)


Epoch [008/10]  Train Loss: 0.2144  Acc: 1.000 | Val Loss: 0.0000  Acc: 1.000  (213.2s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt
Early stopping triggered at epoch 8.

Best validation loss: 0.0000


In [7]:
# ── Plot training curves ──────────────────────────────────────────
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], label='Train', color='steelblue', linewidth=2)
ax1.plot(epochs, history['val_loss'],   label='Val',   color='tomato',    linewidth=2, linestyle='--')
ax1.set_title('Loss Curve'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_acc'], label='Train', color='seagreen', linewidth=2)
ax2.plot(epochs, history['val_acc'],   label='Val',   color='darkorange', linewidth=2, linestyle='--')
ax2.set_title('Accuracy Curve'); ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('VL-JEPA Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nBest Val Acc  : {max(history["val_acc"]):.3f}')
print(f'Best Val Loss : {min(history["val_loss"]):.4f}')


Best Val Acc  : 1.000
Best Val Loss : 0.0000


/var/folders/td/71tf1gdd2_q3mzb3cs80h1lm0000gn/T/ipykernel_22374/3057016035.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
